# Etapa 1 — Framework físico de waterflooding

**Modelo de Machine Learning para el cribado técnico de reservorios candidatos a waterflooding**

Proyecto de titulación — Maestría en Petróleos, ESPOL
En cooperación con EP Petroecuador (Gerencia de Activo Auca)

---

Este cuaderno corresponde a la **primera casilla de la Etapa 1** del flujo de trabajo
metodológico: *Desarrollo y verificación del framework físico de waterflooding*.

El framework combina tres modelos clásicos de ingeniería de yacimientos:

| Modelo | Aporte al framework | Referencia |
|---|---|---|
| Permeabilidades relativas de Corey | Curvas `kro(Sw)` y `krw(Sw)` | Corey (1954); Brooks & Corey (1964) |
| Desplazamiento de Buckley-Leverett | Flujo fraccional, frente de choque, recobro | Buckley & Leverett (1942) |
| Construcción de Welge | Saturación promedio e irrupción (breakthrough) | Welge (1952) |
| Ley de Darcy (radial) | Relación de movilidad e inyectividad | Craig (1971); Willhite (1986) |

**Propósito.** El framework NO pretende sustituir un simulador numérico de yacimientos.
Su función es generar, de forma rápida y físicamente consistente, las características
derivadas (factor de recobro, relación de movilidad, tiempo de irrupción, inyectividad)
que alimentarán el dataset sintético de entrenamiento del modelo de Machine Learning.

**Supuestos.** Flujo 1-D, bifásico (agua-petróleo), incompresible, inmiscible;
se desprecian la presión capilar y los efectos gravitacionales. Estos son los
supuestos estándar de la teoría de Buckley-Leverett para cribado técnico.

## 1. Configuración del entorno

La celda siguiente funciona tanto en **Google Colab** como en local.

En Colab clona el repositorio para poder importar los módulos de `src/physics/`.
Mientras el repositorio sea **privado**, se requiere un token personal de GitHub:
guárdalo en el panel de *Secrets* de Colab (icono de llave 🔑 en la barra izquierda)
con el nombre `GITHUB_TOKEN`. Cuando el repositorio se haga público al momento de
enviar el artículo, el clonado funcionará sin token y esta celda seguirá siendo válida.

In [ ]:
import os
import sys
import subprocess

GITHUB_USER = "phabelog"
REPO_NAME = "waterflooding-ml-screening"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB and not os.path.exists(f"/content/{REPO_NAME}"):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass

    if token:
        url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    else:
        # Repositorio público: no se necesita token
        url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    subprocess.run(["git", "clone", "-q", url, f"/content/{REPO_NAME}"], check=True)
    print("Repositorio clonado.")

# Localizar la carpeta src/physics tanto en Colab como en local
if IN_COLAB:
    PHYSICS_PATH = f"/content/{REPO_NAME}/src/physics"
    REPO_ROOT = f"/content/{REPO_NAME}"
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    PHYSICS_PATH = os.path.join(REPO_ROOT, "src", "physics")

sys.path.insert(0, PHYSICS_PATH)
print("Ruta de módulos físicos:", PHYSICS_PATH)
print("¿Existe?:", os.path.isdir(PHYSICS_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from corey import relative_permeabilities, normalized_saturation
from buckley_leverett import fractional_flow, welge_results
from darcy import endpoint_mobility_ratio, radial_injectivity_index

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
print("Módulos importados correctamente.")

## 2. Caso base de control

Para verificar el framework se define un reservorio de control con propiedades
típicas de una arenisca consolidada. Los resultados de este caso se contrastan
contra el comportamiento esperado por la teoría.

In [ ]:
caso_base = dict(
    Swi=0.20,      # saturación de agua irreducible [fracción]
    Sor=0.25,      # saturación residual de petróleo [fracción]
    kro_max=0.80,  # kro en Sw = Swi (end-point)
    krw_max=0.30,  # krw en Sw = 1 - Sor (end-point)
    no=2.0,        # exponente de Corey, petróleo
    nw=2.0,        # exponente de Corey, agua
    muo=5.0,       # viscosidad del petróleo [cP]
    muw=1.0,       # viscosidad del agua [cP]
)

for k, v in caso_base.items():
    print(f"  {k:8s} = {v}")

## 3. Curvas de permeabilidad relativa (Corey)

El modelo de Corey expresa las permeabilidades relativas en función de la
saturación de agua normalizada:

$$S_{wn} = \frac{S_w - S_{wi}}{1 - S_{wi} - S_{or}}$$

$$k_{rw} = k_{rw}^{max} \, S_{wn}^{\,n_w} \qquad
k_{ro} = k_{ro}^{max} \, (1 - S_{wn})^{\,n_o}$$

In [ ]:
Sw = np.linspace(caso_base["Swi"], 1 - caso_base["Sor"], 300)
kro, krw = relative_permeabilities(
    Sw, caso_base["Swi"], caso_base["Sor"],
    caso_base["kro_max"], caso_base["krw_max"],
    caso_base["no"], caso_base["nw"],
)

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(Sw, kro, color="#1f4e79", lw=2, label=r"$k_{ro}$")
ax.plot(Sw, krw, color="#2e8b57", lw=2, label=r"$k_{rw}$")
ax.axvline(caso_base["Swi"], color="gray", ls=":", lw=1)
ax.axvline(1 - caso_base["Sor"], color="gray", ls=":", lw=1)
ax.text(caso_base["Swi"], 0.85, r" $S_{wi}$", fontsize=9, color="gray")
ax.text(1 - caso_base["Sor"], 0.85, r" $1-S_{or}$", fontsize=9, color="gray", ha="right")
ax.set_xlabel("Saturación de agua, $S_w$")
ax.set_ylabel("Permeabilidad relativa")
ax.set_title("Curvas de permeabilidad relativa (modelo de Corey)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"krw(Swi)     = {krw[0]:.6f}   (debe ser 0)")
print(f"kro(1-Sor)   = {kro[-1]:.6f}   (debe ser 0)")
print(f"kro(Swi)     = {kro[0]:.4f}   (debe ser kro_max = {caso_base['kro_max']})")
print(f"krw(1-Sor)   = {krw[-1]:.4f}   (debe ser krw_max = {caso_base['krw_max']})")

## 4. Curva de flujo fraccional y construcción de Welge

Despreciando gravedad y presión capilar, el flujo fraccional de agua es:

$$f_w = \frac{1}{1 + \dfrac{k_{ro}\,\mu_w}{k_{rw}\,\mu_o}}$$

La **construcción de Welge** traza la tangente a la curva $f_w(S_w)$ desde el punto
$(S_{wi}, 0)$. El punto de tangencia define la saturación del frente de choque $S_{wf}$,
y la pendiente de esa tangente determina los volúmenes porosos inyectados a la
irrupción: $Q_{i,BT} = 1 / (df_w/dS_w)_{S_{wf}}$.

In [ ]:
params_fw = {k: caso_base[k] for k in
              ["Swi", "Sor", "kro_max", "krw_max", "no", "nw", "muo", "muw"]}

res = welge_results(params_fw, caso_base["Swi"], caso_base["Sor"])

print("--- Resultados Buckley-Leverett / Welge ---")
print(f"  Swf  (saturación del frente)        = {res['Swf']:.4f}")
print(f"  fwf  (flujo fraccional en el frente)= {res['fwf']:.4f}")
print(f"  Pendiente de la tangente            = {res['slope']:.4f}")
print(f"  Qi_BT (PV inyectados a irrupción)   = {res['Qi_BT']:.4f}")
print(f"  Sw promedio a la irrupción          = {res['Sw_avg_BT']:.4f}")
print(f"  Factor de recobro a la irrupción    = {res['RF_BT']*100:.2f} %")
print()
print(f"  Verificación cruzada de Swf (dos métodos numéricos independientes):")
print(f"  diferencia = {res['cross_check_diff']:.2e}  (debe ser ~0)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

# --- Panel izquierdo: flujo fraccional + tangente de Welge ---
ax = axes[0]
fwc = res["fw_curve"]
ax.plot(fwc["Sw"], fwc["fw"], color="#1f4e79", lw=2, label=r"$f_w(S_w)$")
Sw_tan = np.array([caso_base["Swi"], 1.0])
ax.plot(Sw_tan, res["slope"] * (Sw_tan - caso_base["Swi"]),
        "--", color="#c0392b", lw=1.5, label="Tangente de Welge")
ax.plot([res["Swf"]], [res["fwf"]], "o", color="#c0392b", ms=8,
        label=f"Frente: $S_{{wf}}$={res['Swf']:.3f}")
ax.plot([res["Sw_avg_BT"]], [1.0], "s", color="#e67e22", ms=7,
        label=f"$\\bar{{S}}_w$ irrupción={res['Sw_avg_BT']:.3f}")
ax.axvline(caso_base["Swi"], color="gray", ls=":", lw=0.8)
ax.axhline(1.0, color="gray", lw=0.5)
ax.set_xlim(0, 1); ax.set_ylim(-0.02, 1.08)
ax.set_xlabel("Saturación de agua, $S_w$")
ax.set_ylabel("Flujo fraccional de agua, $f_w$")
ax.set_title("Flujo fraccional y construcción de Welge")
ax.legend(loc="upper left", fontsize=8.5)
ax.grid(alpha=0.3)

# --- Panel derecho: curva de recobro ---
ax = axes[1]
rc = res["recovery_curve"]
Qi_pre = np.linspace(1e-3, res["Qi_BT"], 60)
RF_pre = Qi_pre / (1.0 - caso_base["Swi"])
Qi_all = np.concatenate((Qi_pre, rc["Qi"]))
RF_all = np.concatenate((RF_pre, rc["RF"]))
order = np.argsort(Qi_all)
ax.plot(Qi_all[order], RF_all[order] * 100, color="#1f4e79", lw=2)
ax.axvline(res["Qi_BT"], color="#c0392b", ls="--", lw=1.2,
           label=f"Irrupción ($Q_i$={res['Qi_BT']:.2f} PV, RF={res['RF_BT']*100:.1f}%)")
ax.set_xscale("log"); ax.set_xlim(1e-2, 20)
ax.set_xlabel("Volúmenes porosos inyectados, $Q_i$ (escala log)")
ax.set_ylabel("Factor de recobro, RF (%)")
ax.set_title("Recobro vs inyección acumulada")
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## 5. Verificación física del framework

Un framework que va a etiquetar miles de casos sintéticos debe demostrar que
reproduce el comportamiento físico esperado. Se verifican cinco condiciones.

In [ ]:
verificaciones = []

# (1) Condiciones de frontera de las permeabilidades relativas
verificaciones.append(("krw(Swi) = 0", np.isclose(krw[0], 0.0, atol=1e-8)))
verificaciones.append(("kro(1-Sor) = 0", np.isclose(kro[-1], 0.0, atol=1e-8)))

# (2) Condiciones de frontera y monotonicidad de fw
verificaciones.append(("fw(Swi) ~ 0", fwc["fw"][0] < 0.05))
verificaciones.append(("fw(1-Sor) ~ 1", fwc["fw"][-1] > 0.95))
verificaciones.append(("fw monótona creciente", bool(np.all(np.diff(fwc["fw"]) >= -1e-6))))

# (3) Coincidencia de los dos métodos de localización del frente
verificaciones.append(("Swf coincide entre métodos A y B", res["cross_check_diff"] < 5e-3))

# (4) Rangos físicos válidos
verificaciones.append(("Swi < Swf < 1-Sor",
                       caso_base["Swi"] < res["Swf"] < 1 - caso_base["Sor"]))
verificaciones.append(("Swf <= Sw_avg_BT <= 1-Sor",
                       res["Swf"] <= res["Sw_avg_BT"] <= 1 - caso_base["Sor"]))
verificaciones.append(("0 <= RF_BT <= 1", 0.0 <= res["RF_BT"] <= 1.0))

print(f"{'Verificación':45s} {'Resultado'}")
print("-" * 60)
for nombre, ok in verificaciones:
    print(f"{nombre:45s} {'PASA' if ok else 'FALLA'}")

fallidas = [n for n, ok in verificaciones if not ok]
print("-" * 60)
print(f"{len(verificaciones) - len(fallidas)}/{len(verificaciones)} verificaciones superadas.")

### 5.1 Sensibilidad a la viscosidad del petróleo

La quinta verificación es de **comportamiento**, no de frontera: al aumentar la
viscosidad del petróleo, la relación de movilidad se vuelve más desfavorable
(mayor $M$), el frente se desestabiliza y el factor de recobro a la irrupción
debe **disminuir**. Si el framework no reprodujera esta tendencia, el etiquetado
apto/no apto del dataset sintético carecería de sentido físico.

In [ ]:
viscosidades = [1, 2, 5, 10, 20, 50, 100]
filas = []

for muo in viscosidades:
    p = dict(params_fw); p["muo"] = muo
    r = welge_results(p, caso_base["Swi"], caso_base["Sor"])
    M = endpoint_mobility_ratio(caso_base["krw_max"], caso_base["muw"],
                                caso_base["kro_max"], muo)
    filas.append((muo, M, r["Swf"], r["Qi_BT"], r["RF_BT"] * 100))

print(f"{'muo [cP]':>9} {'M':>8} {'Swf':>8} {'Qi_BT [PV]':>12} {'RF_BT [%]':>11}")
print("-" * 52)
for muo, M, swf, qi, rf in filas:
    print(f"{muo:>9} {M:>8.3f} {swf:>8.4f} {qi:>12.4f} {rf:>11.2f}")

rf_series = [f[4] for f in filas]
monotona = all(rf_series[i] > rf_series[i + 1] for i in range(len(rf_series) - 1))
print("-" * 52)
print(f"RF decrece monótonamente al aumentar muo: {'PASA' if monotona else 'FALLA'}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(6.5, 4.2))

M_vals = [f[1] for f in filas]
ax1.plot(M_vals, rf_series, "o-", color="#1f4e79", lw=2, ms=6)
ax1.set_xscale("log")
ax1.set_xlabel("Relación de movilidad end-point, $M$ (escala log)")
ax1.set_ylabel("Factor de recobro a la irrupción, RF (%)", color="#1f4e79")
ax1.tick_params(axis="y", labelcolor="#1f4e79")
ax1.axvline(1.0, color="#c0392b", ls="--", lw=1.2)
ax1.text(1.05, min(rf_series) + 2, " M = 1\n(límite favorable)",
         color="#c0392b", fontsize=8.5)
ax1.set_title("Efecto de la relación de movilidad sobre el recobro")
ax1.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## 6. Ley de Darcy: movilidad e inyectividad

Dos indicadores adicionales alimentarán el dataset sintético:

**Relación de movilidad de punto extremo** (Craig, 1971):

$$M = \frac{k_{rw}^{max}/\mu_w}{k_{ro}^{max}/\mu_o}$$

Valores $M < 1$ indican desplazamiento favorable (frente estable);
$M > 1$ advierte riesgo de digitación viscosa y canalización temprana del agua.

**Índice de inyectividad radial** (Darcy, estado estacionario, unidades de campo):

$$II = \frac{0.00708\,k\,h}{\mu_w\left[\ln(r_e/r_w) - 0.75 + S\right]}
\quad \left[\frac{bbl/d}{psi}\right]$$

In [ ]:
M_base = endpoint_mobility_ratio(caso_base["krw_max"], caso_base["muw"],
                                 caso_base["kro_max"], caso_base["muo"])
print(f"Relación de movilidad del caso base: M = {M_base:.3f}"
      f"  ({'FAVORABLE' if M_base < 1 else 'DESFAVORABLE'})")
print()

print(f"{'k [mD]':>8} {'h [ft]':>8} {'II [bbl/d/psi]':>16} {'q a Δp=500 psi [bbl/d]':>25}")
print("-" * 62)
for k_md, h_ft in [(50, 20), (100, 30), (200, 40), (500, 50), (1000, 60)]:
    II = radial_injectivity_index(k_md=k_md, h_ft=h_ft, muw_cp=caso_base["muw"],
                                  re_ft=1000, rw_ft=0.33, skin=0.0)
    print(f"{k_md:>8} {h_ft:>8} {II:>16.3f} {II*500:>25.0f}")

## 7. Conclusiones de la Etapa 1

1. El framework físico reproduce correctamente las condiciones de frontera del
   modelo de Corey y de la curva de flujo fraccional.
2. La localización del frente de choque fue verificada con **dos métodos numéricos
   independientes** (búsqueda en malla de la máxima pendiente secante y solución de
   la condición de tangencia por el método de Brent), que coinciden dentro de una
   tolerancia de $10^{-5}$. La verificación no es circular.
3. La saturación promedio a la irrupción calculada por **balance de materia**
   coincide con la obtenida por la **construcción de Welge**, lo que confirma la
   consistencia interna del cálculo de recobro.
4. El framework reproduce la tendencia física esperada: a mayor viscosidad del
   petróleo (mayor $M$), menor factor de recobro a la irrupción y menor volumen
   poroso inyectado hasta el breakthrough.
5. Los indicadores de Darcy (relación de movilidad e índice de inyectividad)
   producen magnitudes dentro de rangos operativos realistas.

**El framework queda validado para su uso en la generación del dataset sintético.**

---

### Siguiente paso metodológico

Casilla 2 de la Etapa 1: *Definición de variables de entrada y sus rangos*,
seguida del *cálculo de características físicas derivadas y etiquetado apto/no apto*.
Esa etapa requiere fijar los criterios de aptitud, combinando los rangos de cribado
tradicional de la literatura con las salidas físicas calculadas aquí.